In [ ]:

import pandas.io.formats.style
import pandas as pd
import os
import numpy as np

def write_to_html_file(df, clusters_dict, field_name, filename='out.html'):
    '''
    Write an entire dataframe to an HTML file with nice formatting.
    '''

    result = '''

<HTML>
<HEAD>

    <meta name="viewport" content="width=device-width, height=device-height, initial-scale=1.0, user-scalable=no">
    <script type="text/javascript">
    var hipsDir=null;</script>

    <meta charset="UTF-8">
    <meta http-equiv="X-UA-Compatible"
        content="IE=edge">
    <!-- <meta name="viewport"
        content="width=device-width, 
                initial-scale=1.0"> -->
    <style>
        .popup {
            position: absolute;
            z-index: 1;
            left: 60%;
            top: 0;
            width: 40%;
            height: 100%;
            overflow: auto;
            background-color: rgba(0, 0, 0, 0.4);
            display: none;
        }
        .popup-content {
            background-color: white;
            margin: 10% auto;
            padding: 20px;
            border: 1px solid #888888;
            width: 90%;
            font-weight: bolder;
        }
        .popup-content button {
            display: block;
            margin: 0 auto;
        }
        .show {
            display: block;
        }
        /* h1 {
            color: green;
        } */
    </style>

</HEAD>
'''

    result += f'<H1> Multiwavelength Classification of {field_name} Field using MUWCLASS <button id="myButton"> Project Overview </button> </H1>'

    result += '''

<!-- This Web resource contains HiPS(*) components for <B>ESO EPO</B> progressive survey. -->


<div id="myPopup" class="popup">
  <div class="popup-content">
    <h2>Project goals</h2>
      <p>We plan to develop an automated multiwavelength machine-learning classification pipeline (MUWCLASS) to identify the nature of X-ray sources 
    that have been observed by <a href='https://cxc.cfa.harvard.edu/csc/'>Chandra Source Catalog v2.0</a>. 

      <h2>Work performed</h2>
      <p>We collected  multwavelength properties of all X-ray sources in these fields using the Chandra X-ray Source catalog and multiwavelength properties from  Gaia-DR3, 2MASS,
        and WISE catalogs. We used the machine-learning approach implemented in our MUWCLASS pipeline (see e.g., <a href='https://ui.adsabs.harvard.edu/abs/2022ApJ...941..104Y/abstract'>Yang et al., 2022</a>).
        The pipeline relies on the training dataset of ~3,000 X-ray sources that have known astrophysical types (currently these are LM-STAR, HM-STAR, YSO, AGN, LMXB, HMXB, CV, NS).
        We used this pipeline to classify X-ray sources whose classifications are provided in the table below.</p>
      <h2>Funding support</h2>
      <p>This work is supported by the National Aeronautics and Space Administration (NASA) through the Astrophysics Data Analysis Program (ADAP) award 80NSSC19K0576.</p>

      <button id="closePopup">
            Close
        </button>
  </div>
</div>



<script>
    myButton.addEventListener("click", function () {
        myPopup.classList.add("show");
    });
    closePopup.addEventListener("click", function () {
        myPopup.classList.remove("show");
    });
    window.addEventListener("click", function (event) {
        if (event.target == myPopup) {
            myPopup.classList.remove("show");
        }
    });
</script>




<TABLE>
<TR>
<TD>
<script src="https://code.jquery.com/jquery-1.10.1.min.js"></script>
<script type="text/javascript" src="aladin.js" charset="utf-8"></script>

<div id="aladin-lite-div" style="width:900px;height:600px"> 
  <div style="background-color: rgba(255, 255, 255, 0.6); z-index: 20; position: absolute; left: 10px;bottom: 30%;">&nbsp;&nbsp;Overlay opacity:<br><input id="opacity" type="range" min="0" max="1" step="0.05" value="0.5"></div>
</div>


Show sources with CT greater than:
<input id='slider' style='vertical-align:middle;width:40vw;' step='0.1' min='0' max='10' type='range' value='0'>
<span id='pmVal'  >0 </span><br><br><div id='aladin-lite-div' style='width: 400px;height: 5px;'></div>

and with Significance greater than:
<input id='slider2' style='vertical-align:middle;width:40vw;' step='0.1' min='3' max='10' type='range' value='0'>
<span id='sigVal'  >0 </span><br><br><div id='aladin-lite-div' style='width: 400px;height: 5px;'></div>



<script type="text/javascript">

    const slider = document.getElementById('opacity');
    slider.oninput = function() {
        aladin.getOverlayImageLayer().setAlpha(slider.value);
    };


    let aladin;
    A.init.then(() => {

        var pmThreshold = 0;
        var sigThreshold = 0;

        var slider = document.getElementById('slider');
        slider.oninput = function() {
            pmThreshold = this.value;
            $('#pmVal').html(pmThreshold);
            hips.reportChange();
        }

        var sigThreshold = 0;

        var slider = document.getElementById('slider2');
        slider.oninput = function() {
            sigThreshold = this.value;
            $('#sigVal').html(sigThreshold);
            hips.reportChange();
        }
        
        var myFilterFunction = function(source) {
            var totalPm  = parseFloat(source.data['CT']);
            var signif  = parseFloat(source.data['significance']);
            if (isNaN(totalPm) || isNaN(signif)) {
                return false;
            }
            return totalPm>pmThreshold && signif>sigThreshold;
        }

        // define custom draw function
        var drawFunction = function(source, canvasCtx, viewParams) {
            canvasCtx.beginPath();
            canvasCtx.arc(source.x, source.y,6, 0, 2 * Math.PI, false);
            canvasCtx.closePath();
            // canvasCtx.strokeStyle = '#c38';
            canvasCtx.strokeStyle = source.data['color'];
            canvasCtx.lineWidth = 3;
            canvasCtx.globalAlpha = 0.7,
            canvasCtx.stroke();
            var fov = Math.max(viewParams['fov'][0], viewParams['fov'][1]);

            // object name is displayed only if fov<10°
            if (fov>10) {
                return;
            }

            canvasCtx.globalAlpha = 0.9;
            canvasCtx.globalAlpha = 1;

            var xShift = 20;

            canvasCtx.font = '15px Arial'
            canvasCtx.fillStyle = '#eee';
            // canvasCtx.fillText(source.data['name'], source.x + xShift, source.y -4);

            // object type is displayed only if fov<2°
            if (fov>0.05) {
                return;
            }
            canvasCtx.font = '12px Arial'
            canvasCtx.fillStyle = '#abc';
            canvasCtx.fillText(source.data['Class'], source.x + 2 + xShift, source.y + 10);
        };

        // define custom draw function
        var drawFunction2 = function(source, canvasCtx, viewParams) {
            canvasCtx.beginPath();
            // canvasCtx.arc(source.x, source.y,10, 0, 2 * Math.PI, false);
            canvasCtx.rect(source.x-8, source.y-8,16,16)
            canvasCtx.closePath();
            // canvasCtx.strokeStyle = '#c38';
            canvasCtx.strokeStyle = source.data['color'];
            canvasCtx.lineWidth = 3;
            canvasCtx.globalAlpha = 0.7,
            canvasCtx.stroke();
            var fov = Math.max(viewParams['fov'][0], viewParams['fov'][1]);

            // object name is displayed only if fov<10°
            if (fov>10) {
                return;
            }

            canvasCtx.globalAlpha = 0.9;
            canvasCtx.globalAlpha = 1;

            var xShift = 20;

            canvasCtx.font = '15px Arial'
            canvasCtx.fillStyle = '#eee';
            // canvasCtx.fillText(source.data['name'], source.x + xShift, source.y -4);

            // object type is displayed only if fov<2°
            if (fov>0.05) {
                return;
            }
            canvasCtx.font = '12px Arial'
            canvasCtx.fillStyle = '#abc';
            canvasCtx.fillText(source.data['Class'], source.x + 2 + xShift, source.y + 10);
        };
'''

    result += f'    aladin = A.aladin("#aladin-lite-div", \u007bsurvey: "https://yichaolin-astro.github.io/HESS-J1702-420A/HESS_J1702_420A_Skymap/",showSimbadPointerControl: true, name:"Chandra",  target: "{clusters_dict[field_name]["ra"]} {clusters_dict[field_name]["dec"]}", fov: 12 / 60. \u007d);'

    result += '''
    //aladin.toggleFullscreen();

    aladin.setOverlayImageLayer('https://alasky.cds.unistra.fr/pub/10.1051_0004-6361_201732098flux/'); // CDS/P/HGPS/Flux
    aladin.getOverlayImageLayer().setAlpha(0.5);

    aladin.setOverlayImageLayer('CDS/P/DECaPS/DR2/color'); // https://alasky.cds.unistra.fr/DECaPS/DR2/CDS_P_DECaPS_DR2_color/
    aladin.getOverlayImageLayer().setAlpha(0.5);

    aladin.setOverlayImageLayer('CSIRO/P/RACS/low/I'); // https://casda.csiro.au/hips/RACS/low/I/
    aladin.getOverlayImageLayer().setAlpha(0.5);

    aladin.setOverlayImageLayer('CSIRO/P/RACS/mid/I');
    aladin.getOverlayImageLayer().setAlpha(0.5);

    aladin.setOverlayImageLayer('http://archive-new.nrao.edu/vlass/HiPS/VLASS_Epoch1/Quicklook/');
    aladin.getOverlayImageLayer().setAlpha(0.5);

    // aladin.setOverlayImageLayer('http://cade.irap.omp.eu/documents/Ancillary/4Aladin/CGPS_VGPS/');
    // aladin.getOverlayImageLayer().setAlpha(0.5);

    aladin.setOverlayImageLayer('CDS/P/NVSS'); // https://alasky.cds.unistra.fr/NVSS/intensity/
    aladin.getOverlayImageLayer().setAlpha(0.5);
    
    aladin.setOverlayImageLayer('CDS/P/VISTA/VVV/DR4/ColorJYZ'); // https://alasky.cds.unistra.fr/VISTA/VVV_DR4/VISTA-VVV-DR4-ColorJYZ/
    aladin.getOverlayImageLayer().setAlpha(0.5);

    aladin.setOverlayImageLayer('CDS/P/SPITZER/color'); // https://alasky.cds.unistra.fr/Spitzer/SpitzerI1I2I4color/
    aladin.getOverlayImageLayer().setAlpha(0.5)

      
    
    var overlay = A.graphicOverlay({color: 'white', lineWidth: 3});
        aladin.addOverlay(overlay);
        overlay.addFootprints([
'''
    result += f'            A.ellipse({clusters_dict[field_name]["ra"]}, {clusters_dict[field_name]["dec"]}, {clusters_dict[field_name]["radius"]/(60.*np.cos(clusters_dict[field_name]["dec"]*np.pi/180))},{clusters_dict[field_name]["radius"]/60.},0, \u007bcolor: "cyan"\u007d),'
    
    result += '''
            ]);
        // overlay.add(); // radius in degrees , use the polygon to use the calculated coordinates of ellipses

        var cat = A.catalog({name:'popup', sourceSize: 100, onClick: 'showTable', shape: drawFunction, filter: myFilterFunction}); 
        aladin.addCatalog(cat);
'''
    
    df['color'] = 'red'

    for clas, color in zip(['HM-STAR','AGN','YSO','LMXB','CV','HMXB','LM-STAR','NS'], ['deepskyblue', 'cyan','lime','orange','blue','peru','yellow','magenta']):
        df.loc[df['Class']==clas, 'color'] = color 


    for i, df_s in df.iterrows():
        # print(df_s)
        if df_s['name'][-1]=='0':
            
            result += f'        cat.addSources([A.source({df_s["CSC_RA"]}, {df_s["CSC_DEC"]}, \u007bname:"{df_s["name"]}",CT:{df_s["CT"]},significance:{df_s["significance"]},Class:"{df_s["Class"]}",color:"{df_s["color"]}"\u007d)]);\n'
        else:
            
            result += f"        cat.addSources([A.source({df_s['MW_RA']}, {df_s['MW_DEC']}, \u007bname:'{df_s['name']}',CT:{df_s['CT']},significance:{df_s['significance']},Class:'{df_s['Class']}',color:'{df_s['color']}'\u007d)]);\n"
    
    result += '''
        // cat.addSources([A.source(134.782442, -43.707852, {name:'M 86',CT:5,significance: 8,Class: 'AGN',color: 'lime'})]);

        var hips = A.catalogFromURL('https://raw.githubusercontent.com/huiyang-astro/FGL-aladin-lite-test/main/FGL_11152023_all_class_color.vot', {onClick: 'showTable', sourceSize: 100,  name: 'CSC',shape: drawFunction,filter: myFilterFunction}); //,displayLabel: true, labelColumn: 'Class', labelColor: 'cyan', labelFont: '20px sans-serif', onClick: 'showTable',
        hips.hide();  
        aladin.addCatalog(hips);
        
        $('input[type=radio][name=otype]').change(function() {
            requestedOtype = this.value;
            hips.reportChange();
        });

        var hips2 = A.catalogFromURL('https://raw.githubusercontent.com/huiyang-astro/FGL-aladin-lite-test/main/CSCv2_TD.vot', {onClick: 'showPopup', sourceSize: 200,  name:'TD',shape:drawFunction2}); //,displayLabel: true, labelColumn: 'Class', labelColor: 'cyan', labelFont: '20px sans-serif', onClick: 'showTable',
         aladin.addCatalog(hips2);
        
        $('input[type=radio][name=otype]').change(function() {
            requestedOtype = this.value;
            hips2.reportChange();
        });

        var hips9 = A.catalogFromURL('https://raw.githubusercontent.com/huiyang-astro/FGL-aladin-lite-test/main/ATNF_NS.vot', {onClick: 'showPopup', sourceSize: 20,  name:'ATNF',shape:'plus', color:'magenta'}); //,displayLabel: true, labelColumn: 'Class', labelColor: 'cyan', labelFont: '20px sans-serif', onClick: 'showTable',
         aladin.addCatalog(hips9);
        
        var hips10 = A.catalogFromURL('https://raw.githubusercontent.com/huiyang-astro/FGL-aladin-lite-test/main/TeVCat.vot', {onClick: 'showPopup', sourceSize: 20,  name:'TeVCat',shape:'plus', color:'gold'}); //,displayLabel: true, labelColumn: 'Class', labelColor: 'cyan', labelFont: '20px sans-serif', onClick: 'showTable',
         aladin.addCatalog(hips10);

        // var hips11 = A.catalogFromURL('https://raw.githubusercontent.com/huiyang-astro/FGL-aladin-lite-test/main/BeStar.vot', {onClick: 'showPopup', sourceSize: 20,  name:'BeStar',shape:'plus', color:'rgb(0,255,0)'}); //,displayLabel: true, labelColumn: 'Class', labelColor: 'cyan', labelFont: '20px sans-serif', onClick: 'showTable',
        //  aladin.addCatalog(hips11);

        
    });

    
</script>    
</TD>
<TD>
    

    <button id="readmeButton"> README </button>

    <div id="readmePopup" class="popup">
        <div class="popup-content">
            <h2>Aladin-Lite Visualization Window</h2>
                <!-- <p> -->
            <ol>
                <li>You can filter on the classification confidence threshold (CT, the higher the value is the more confidence is the classification, a default value CT=2 is used for confident classifications) and the X-ray source significance by moving the slidebars below the visualization plot. </li>
                <li>You can select image layers and catalog markers from the Manage Layers Button. You can also use the Simbad pointer Button to search for nearby Simbad sources.  More details on <a href='https://aladin.cds.unistra.fr/AladinLite/doc/'>Aladin Lite documentation page</a>.,</li>
                <!-- <li></li> -->
            </ol> 
            <!-- </p> -->
            <h2>Summary Panel</h2>
            <p>The (Classification) Summary panel shows a summarized information of classifications for all sources and confident classifications of significant sources.</p>
            
            <h2>Interactive Table</h2>
            <ol>
                <li> All X-ray sources without significance or CT cuts will be present in the table.</li>
                <li> You can choose to show more columns by selecting the column names from the Column visibillity button.</li>
            </ol>
            Column definitions:
            <ul>
                <li>name: X-ray name from CSCv2.0, -0 indicates no counterpart, -i indicates different counterparts</li>
                <li>Class: classification from MUWCLASS</li>
                <li>Class_prob,Class_prob_e: classification probability and its uncertainty</li>
                <li>CT: classification confidence threshold</li>
                <li>CSC_RA,CSC_DEC,CSC_err_r0,CSC_err_r1,CSC_PA: X-ray coordinate and its error ellipse in arcsec</li>
                <li>significance: X-ray significance</li>
                <li>Fcsc_{b,s,m,h}: X-ray band fluxes at the broad (0.5-7 keV), soft (0.5-1.2 keV), medium (1.2-2 keV), hard (2-7 keV) bands</li>
                <li>HR_hms: X-ray hardness ratios</li>
                <li>var_intra_prob, var_inter_prob: X-ray intra-observation (within one observation) and inter-observation (between observation) variability probability </li>
                <li>p_any,p_i: NWAY association probability and the probability of each ith counterpart </li>
                <li>MW_RA,MW_DEC,MW_err0,MW_sep:counterpart coordinate, its positional uncertainty (in arcsec), and separation to the X-ray position</li>
                <li>Gaia_DR3Name,CATWISE_Name,AllWISE_Name,TMASS_Name</li>
                <li>Gmag,BPmag,RPmag: Gaia DR3 magnitudes </li>
                <li>RPlx: Gaia DR3 Parallax divided by its standard error </li>
                <li>PM: Gaia DR3 Total proper motion, in arcsec/yr</li>
                <li>rgeo: Gaia EDR3 geometric distance posterior, in pc</li>
                <li>Jmag,Hmag,Kmag: 2MASS magnitudes </li>
                <li>W1mag,W2mag,W3mag: WISE magnitudes </li>
                <li>color:marker color of different classification in the interactive plot</li>
            </ul>
            <!-- <p>This work is supported by the National Aeronautics and Space Administration (NASA) through the Astrophysics Data Analysis Program (ADAP) award 80NSSC19K0576.</p>
     -->
            <button id="closereadmePopup">
                Close
            </button>
        </div>
    </div>

    <script>
        readmeButton.addEventListener("click", function () {
            readmePopup.classList.add("show");
        });
        closereadmePopup.addEventListener("click", function () {
            readmePopup.classList.remove("show");
        });
        window.addEventListener("click", function (event) {
            if (event.target == readmePopup) {
                readmePopup.classList.remove("show");
            }
        });
    </script>

    <div>
        <h3><u>(Classification) Summary</u></h3>    
'''
    class_dict = dict(df['Class'].value_counts())
    df_conf = df[(df['significance']>=5.) & (df['CT']>=2.)]
    class_conf_dict = dict(df_conf['Class'].value_counts())
    class_all = ''
    class_conf = ''
    for clas in df['Class'].unique():
        class_all += f'{class_dict[clas]}{clas} '
    for clas in df_conf['Class'].unique():
        class_conf += f'{class_conf_dict[clas]}{clas} '
    result += f'        <li id="all_clas">All classifications: {class_all}</li>\n'
    result += f'        <li id="conf_clas">Classifications of significant sources (S/N>=5) with CT>=2: {class_conf}</li>'

    result += '''
        <li id="comments">Comment: </li>
      </div>



</TD>
</TR>
</TABLE>


<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Interactive Table with Toggleable Columns</title>

<!-- Include jQuery -->
<script src="https://code.jquery.com/jquery-3.6.4.min.js"></script>

<!-- Include DataTables CSS and JS -->
<link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.10.25/css/jquery.dataTables.css">
<script type="text/javascript" charset="utf8" src="https://cdn.datatables.net/1.10.25/js/jquery.dataTables.js"></script>

<!-- Include DataTables ColumnVisibility extension CSS and JS -->
<link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/buttons/1.7.1/css/buttons.dataTables.min.css">
<script type="text/javascript" charset="utf8" src="https://cdn.datatables.net/buttons/1.7.1/js/dataTables.buttons.min.js"></script>
<script type="text/javascript" charset="utf8" src="https://cdn.datatables.net/buttons/1.7.1/js/buttons.colVis.min.js"></script>


    '''
    
    df = df.replace({-99.:np.nan,'nan':'','NaN':''})
    # print(df_html.to_html())


    formats = {'Class_prob': '{:.2f}','Class_prob_e':'{:.2f}','CT':'{:.2f}','CSC_err_r0':'{:.2f}','Fcsc_s':'{:.2e}','Fcsc_m':'{:.2e}','Fcsc_h':'{:.2e}','Fcsc_b':'{:.2e}','HR_hms':'{:.2f}','p_any':'{:.2f}','p_i':'{:.2f}','MW_err0':'{:.2f}','MW_sep':'{:.2f}','Gmag':'{:.3f}','BPmag':'{:.3f}','RPmag':'{:.3f}','RPlx':'{:.1f}','PM':'{:.5f}','rgeo':'{:.0f}'}

    for col, f in formats.items():
        df[col] = df[col].map(lambda x: f.format(x))
    
    # df['Class'] = df.apply(lambda r: f'<a class="image-link" href="./plot/{r["name"][5:]}_{r.Class}.png" target="_blank">{r.Class}</a>', axis=1)
    
    if type(df) == pd.io.formats.style.Styler:
        result += df.render()
    else:
        result += df.to_html(classes='wide',escape=False,index=False)
    result += '''

<script>
  $(document).ready(function() {
      // Initialize DataTable
      var table = $('#interactiveTable').DataTable({
          dom: 'Bflrtip',

          // Sort by column 11 (0-based index), ascending
            order: [[9, 'acs']],

          columnDefs: [ //{targets:[1,3], visible: false}],
                    {targets: [ 4,  6,  7,  8, 11, 12, 13, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 32, 34, 35, 37, 38, 40, 41], visible: false }, // Hide the Country column by default (index 2)
                ],
          buttons: ['colvis']
      });
  
      // Define column visibility settings
      // table.column(0).visible(true);

  });
  </script>


</body>
</html>

<HR>
Any comments are welcome by <A HREF="https://github.com/muwclass/MUWCLASS/issues/new">submitting an issue report</A>. 
This visualization tool is displayed by <A HREF="https://aladin.u-strasbg.fr/AladinLite">Aladin Lite</A>. 
<br>
<!-- Funding support: This work is supported by the National Aeronautics and Space Administration (NASA) through the Astrophysics Data Analysis Program (ADAP) award 80NSSC19K0576. -->



</HTML>


'''
    result = result.replace('    <table border="1" class="dataframe wide">', '<table id="interactiveTable" class="display" style="width:100%">')
    with open(filename, 'w') as f:
        f.write(result)



In [20]:

field_name = 'J170147.3-421407'
# dictionary of field name and CSCview cone search parameters. Radius is in arcmins
fields_dict = {
'J170147.3-421407': {'ra': 255.4458333, 'dec': -42.2352778, 'radius': 12}, 
}
data_dir = f'/home/hyang/Research/GWU/codes/MUWCLASS-main/demos/data/{field_name}' # data directory to save the file



In [21]:



df_all = pd.read_csv(f'{data_dir}/{field_name}_class.csv')

# Remove the final "-0", "-1", etc. suffix
df_all["base_name"] = (
    df_all["name"]
    .str.replace(r"-\d+$", "", regex=True)
)

# Assign Src1, Src2, ... to each unique base_name
df_all["Src"] = (
    pd.factorize(df_all["base_name"])[0] + 1
)

df_all["Src"] = (
    "Src" + df_all["Src"].astype(str)
)

df_all['name'] = df_all['name'].str[5:]
df_all['CXO_name'] =df_all['name']
# new_rows = pd.DataFrame({
#     'Src': ['extended', 'eastlobe', 'westlobe']
# })


In [22]:
df_all = df_all[df_all['significance']>=5]
# df_all = pd.concat([df_all, new_rows], ignore_index=True, sort=False)
# 'CSC_err_r1','CSC_PA',
df_all['Spec_results'] = 'Plot'
df_html = df_all[['Src','CXO_name','Class','Class_prob','Class_prob_e','CT','CSC_RA','CSC_DEC','CSC_err_r0','significance','Fcsc_b','Fcsc_s','Fcsc_m','Fcsc_h','HR_hms','var_intra_prob',
       'var_inter_prob','p_any','p_i','MW_RA','MW_DEC','MW_err0','MW_sep','DR3Name_gaia','CATWISE_Name','ALLWISE_AllWISE','TMASS_2MASS','Gmag','BPmag','RPmag','RPlx','PM','rgeo','Jmag','Hmag','Kmag','W1mag','W2mag','W3mag','Spec_results','name']].rename(columns={'DR3Name_gaia':'Gaia_DR3Name','ALLWISE_AllWISE':'AllWISE_Name','TMASS__2MASS':'TMASS_Name'})

write_to_html_file(df_html, fields_dict, field_name, filename=f'./{field_name}_class.html')

os.system(f'open ./{field_name}_class.html') # if it does not work, try to open the ./data/{field_name}/{field_name}_class.html yourself with a brower to view the calssification results 


0

Opening in existing browser session.


In [9]:

df_class_select = df_html.sort_values(by=['significance','CXO_name'],ascending=[False,True]).reset_index(drop=True)

for index, df_src in df_class_select.iterrows():# ['name_nospace'].values:
    
    src = df_src['Src']
    src_short = src#[4:]
    x_name = df_src['CXO_name']
    clas = df_src["Class"]
    print('  <tr>')

    print(f"   <td>{df_src['Src']}</td>")
    print(f"   <td>{df_src['CXO_name']}</td>")
    print(f'   <td><a class="image-link" href="./html/images/{x_name}_{clas}.png" target="_blank">{df_src["Class"]}</a></td>')
         
    print(f'   <td>{df_src["Class_prob"]:.2f}</td>')
    print(f'   <td>{df_src["Class_prob_e"]:.2f}</td>')
    print(f"   <td>{df_src['CT']:.2f}</td>")
    print(f"   <td>{df_src['CSC_RA']:.7f}</td>")
    print(f"   <td>{df_src['CSC_DEC']:.7f}</td>")
    print(f"   <td>{df_src['CSC_err_r0']:.7f}</td>")
    print(f"   <td>{df_src['significance']:.2f}</td>")
    print(f"   <td>{df_src['Fcsc_b']:.2e}</td>")
    print(f"   <td>{df_src['Fcsc_s']:.2e}</td>")
    print(f"   <td>{df_src['Fcsc_m']:.2e}</td>")
    print(f"   <td>{df_src['Fcsc_h']:.2e}</td>")
    print(f"   <td>{df_src['HR_hms']:.2f}</td>")
    print(f"   <td>{df_src['var_intra_prob']:.2f}</td>")
    print(f'   <td><a class="image-link" href="./html/images/{src.lower()}_lc.png" target="_blank">{df_src["var_inter_prob"]:.2f}</a></td>')
            
    print(f"   <td>{df_src['p_any']:.2f}</td>")
    print(f"   <td>{df_src['p_i']:.2f}</td>")
    print(f"   <td>{df_src['MW_RA']:.2f}</td>")
    print(f"   <td>{df_src['MW_DEC']:.2f}</td>")
    print(f"   <td>{df_src['MW_err0']:.2f}</td>")
    print(f"   <td>{df_src['MW_sep']:.2f}</td>")
    print(f"   <td>{df_src['Gaia_DR3Name']}</td>")
    print(f"   <td>{df_src['CATWISE_Name']}</td>")
    print(f"   <td>{df_src['AllWISE_Name']}</td>")
    print(f"   <td>{df_src['TMASS_2MASS']}</td>")
    
    print(f"   <td>{df_src['Gmag']:.2f}</td>")
    print(f"   <td>{df_src['BPmag']:.2f}</td>")
    print(f"   <td>{df_src['RPmag']:.2f}</td>")
    print(f"   <td>{df_src['RPlx']:.2f}</td>")
    print(f"   <td>{df_src['PM']:.2f}</td>")
    print(f"   <td>{df_src['rgeo']:.2f}</td>")
    print(f"   <td>{df_src['Jmag']:.2f}</td>")
    print(f"   <td>{df_src['Hmag']:.2f}</td>")
    print(f"   <td>{df_src['Kmag']:.2f}</td>")
    print(f"   <td>{df_src['W1mag']:.2f}</td>")
    print(f"   <td>{df_src['W2mag']:.2f}</td>")
    print(f"   <td>{df_src['W3mag']:.2f}</td>")

    print(f'   <td><a class="image-link" href="./html/images/{src.lower()}_pl_spectrum.png" target="_blank">plot</a></td>')
    print(f"   <td>{df_src['name']}</td>")
    print(f"   <td>{df_src['color']}</td>")




    print('  </tr>')    


    



        
       
        
        

  <tr>
   <td>Src1</td>
   <td>J170157.2-421027-0</td>
   <td><a class="image-link" href="./html/images/J170157.2-421027-0_NS.png" target="_blank">NS</a></td>
   <td>0.75</td>
   <td>0.15</td>
   <td>2.30</td>
   <td>255.4882177</td>
   <td>-42.1740326</td>
   <td>0.2754051</td>
   <td>48.15</td>
   <td>1.36e-13</td>
   <td>4.11e-15</td>
   <td>2.70e-15</td>
   <td>1.29e-13</td>
   <td>0.90</td>
   <td>0.96</td>
   <td><a class="image-link" href="./html/images/src1_lc.png" target="_blank">0.88</a></td>
   <td>0.00</td>
   <td>0.00</td>
   <td>-99.00</td>
   <td>-99.00</td>
   <td>-99.00</td>
   <td>-99.00</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td>nan</td>
   <td><a class="image-link" href="./html/images/src1_pl_spectrum.png" target="_blank">plot</a></td>
   <td>J170157.2-421027-0

[8573:8604:0904/152909.823012:ERROR:google_apis/gcm/engine/registration_request.cc:290] Registration response error message: DEPRECATED_ENDPOINT


In [ ]:

df = pd.read_csv(f'{data_dir}/{field_name}_class.csv')
df['rgeo_plus'] = df['B_rgeo'] - df['rgeo']
df['rgeo_minus'] =  df['rgeo'] - df['b_rgeo']
df['F_b_e14'] = df['F_b']*1e14

# Remove the final "-0", "-1", etc. suffix
df["base_name"] = (
    df["name"]
    .str.replace(r"-\d+$", "", regex=True)
)

# Assign Src1, Src2, ... to each unique base_name
df["Src"] = (
    pd.factorize(df["base_name"])[0] + 1
)

df["Src"] = (
    "Src" + df["Src"].astype(str)
)

def safe_fmt(x, fmt):
    """Return formatted value or empty string if NaN"""
    return "" if pd.isna(x) else format(x, fmt)

def sci_fmt(x):
    """Scientific notation like x.xxexx"""
    return "" if pd.isna(x) else f"{x:.2e}"

def pm_fmt(val, err, precision=2):
    """Format val ± err or empty if val is NaN"""
    if pd.isna(val):
        return ""
    if pd.isna(err):
        return f"${val:.{precision}f}$"
    return f"${val:.{precision}f} \\pm {err:.{precision}f}$"

def rgeo_fmt(row):
    """Format rgeo^{+plus}_{-minus}"""
    if pd.isna(row['rgeo']):
        return ""

    plus = "" if pd.isna(row['rgeo_plus']) else f"+{row['rgeo_plus']:.0f}"
    minus = "" if pd.isna(row['rgeo_minus']) else f"-{row['rgeo_minus']:.0f}"

    return f"${row['rgeo']:.0f}^{{{plus}}}_{{{minus}}}$"

latex_df = pd.DataFrame({
    "Source": df['Src'], 
    # "Name": df["name"].fillna(""),
    "Class": df["Class"].fillna(""),

    "Class Prob": [
        pm_fmt(v, e, 2)
        for v, e in zip(df["Class_prob"], df["Class_prob_e"])
    ],

    "CT": df["CT"].map(lambda x: safe_fmt(x, ".2f")),
    "RA": df["ra"].map(lambda x: safe_fmt(x, ".5f")),
    "Dec": df["dec"].map(lambda x: safe_fmt(x, ".5f")),
    "PU": df["PU"].map(lambda x: safe_fmt(x, ".2f")),
    "Significance": df["significance"].map(lambda x: safe_fmt(x, ".1f")),
    "F_b": df["F_b_e14"].map(lambda x: safe_fmt(x, ".1f")),#.map(sci_fmt),
    "HR_hms": df["HR_hms"].map(lambda x: safe_fmt(x, ".2f")),
    "var_intra_prob": df["var_intra_prob"].map(lambda x: safe_fmt(x, ".3f")),
    "var_inter_prob": df["var_inter_prob"].map(lambda x: safe_fmt(x, ".3f")),
    "p_any": df["p_any"].map(lambda x: safe_fmt(x, ".2f")),
    "p_i": df["p_i"].map(lambda x: safe_fmt(x, ".2f")),
    # "rgeo": df.apply(rgeo_fmt, axis=1),
    "Gmag": df["Gmag"].map(lambda x: safe_fmt(x, ".2f")),
    
    "Jmag": df["Jmag"].map(lambda x: safe_fmt(x, ".2f")),
    "W1mag": df["W1mag"].map(lambda x: safe_fmt(x, ".2f"))
})

latex_table = latex_df.to_latex(
    index=False,
    escape=False,
    na_rep=""
)

print(latex_table)